# 50 - TF-IDF para clusterizacao baseline

Este notebook compara representacoes TF-IDF para preparar a primeira clusterizacao da base processada filtrada. A unidade de analise e uma descricao normalizada unica, com `frequencia` preservada para analise e uso posterior como peso.

**Branch experimental `notas-pessoais`:** a entrada e a base `notas-pessoais_semstopwords` (notas fiscais pessoais do autor, nov/2025 a nov/2026), definida diretamente neste notebook em vez das constantes de `nofis_classifier.datasets`. Os relatorios sao gravados em `reports/tfidf/notas-pessoais/` para nao sobrescrever os da base original.

Esta celula importa as funcoes reutilizaveis do pacote. O notebook fica responsavel pela narrativa da pesquisa, enquanto `datasets.py` cuida da base e `tfidf.py` concentra as configuracoes e execucoes de TF-IDF.

In [1]:
import pandas as pd

from nofis_classifier.datasets import (
    load_processed_items,
    prepare_modeling_items,
    processed_dataset_path,
)
from nofis_classifier.paths import find_project_root
from nofis_classifier.tfidf import (
    BASELINE_TFIDF_CONFIG_NAME,
    default_tfidf_configs,
    run_tfidf_experiments,
    save_tfidf_reports,
)

pd.set_option("display.max_colwidth", 120)

Aqui sao definidos os caminhos do experimento: a raiz do projeto, o Parquet processado que sera usado como entrada e a pasta onde os relatorios de TF-IDF serao gravados.

In [2]:
DATASET_NAME = "notas-pessoais_semstopwords"
PERIOD_LABEL = "202511-202611"

PROJECT_ROOT = find_project_root()
DATASET_PATH = processed_dataset_path(
    dataset_name=DATASET_NAME,
    period_label=PERIOD_LABEL,
    root=PROJECT_ROOT,
)
REPORT_DIR = PROJECT_ROOT / "reports" / "tfidf" / "notas-pessoais"

DATASET_PATH

WindowsPath('C:/Users/samuel.simao/Sources/Nofis-Classifier/data/processed/notas-pessoais_semstopwords/notas-pessoais_semstopwords-202511-202611.parquet')

## Carregar e validar base

A base filtrada ja deve estar consolidada por `descricao_normalizada`. A preparacao abaixo valida colunas obrigatorias, rejeita duplicatas e garante que a frequencia total nao foi alterada.

Esta celula transforma a base processada na base de modelagem. Para o TF-IDF, cada `descricao_normalizada` unica vira um documento; a coluna `frequencia` e preservada para analises posteriores, mas nao altera diretamente o ajuste do vetor TF-IDF nesta etapa.

In [3]:
raw_items = load_processed_items(DATASET_PATH)
modeling_items = prepare_modeling_items(raw_items)

assert modeling_items["descricao_normalizada"].is_unique
assert modeling_items["descricao_normalizada"].str.strip().ne("").all()
assert raw_items["frequencia"].sum() == modeling_items["frequencia"].sum()

display(modeling_items.head())
display(
    pd.DataFrame(
        {
            "descricoes_unicas": [len(modeling_items)],
            "frequencia_total": [int(modeling_items["frequencia"].sum())],
            "frequencia_media": [modeling_items["frequencia"].mean()],
            "primeiro_mes": [raw_items["primeiro_mes"].min()],
            "ultimo_mes": [raw_items["ultimo_mes"].max()],
        }
    )
)

,descricao_id,descricao_normalizada,frequencia
0,0,10 hot fila adicional,1
1,1,2 hashis 2 molheiras,1
2,2,6 shoyu normal cortesia,1
3,3,abacate kg,1
4,4,abacaxi perola un,16


,descricoes_unicas,frequencia_total,frequencia_media,primeiro_mes,ultimo_mes
0,987,1931,1.956434,2025-11,2026-11


## Comparar configuracoes de TF-IDF

As configuracoes comparam unigramas, unigramas com bigramas e caracteres `char_wb`. A configuracao baseline inicial usa caracteres `(3, 5)` com `min_df=2`, porque descricoes fiscais costumam ter abreviacoes, marcas, erros e variacoes curtas.

Esta e a celula principal da etapa 5 do plano. Ela monta varias configuracoes de TF-IDF e ajusta uma matriz para cada uma, permitindo comparar palavras isoladas, palavras com bigramas, n-gramas de caracteres, `min_df` e `max_features`.

In [4]:
configs = default_tfidf_configs()
summary, vocabulary_samples, vectorizers, matrices = run_tfidf_experiments(modeling_items, configs)

for config in configs:
    matrix = matrices[config.name]
    assert matrix.shape[0] == len(modeling_items)
    assert matrix.shape[1] > 0
    assert matrix.shape[1] <= config.max_features

summary.sort_values(["analyzer", "min_df", "feature_count"])

,name,analyzer,ngram_range,min_df,max_features,max_df,sublinear_tf,norm,document_count,feature_count,non_zero_values,total_cells,density,sparsity
2,char_wb_3_5_min_df_2,char_wb,"(3, 5)",2,30000,0.95,True,l2,987,6727,47599,6639549,0.007169,0.992831
5,char_wb_3_5_min_df_5,char_wb,"(3, 5)",5,30000,0.95,True,l2,987,2743,36909,2707341,0.013633,0.986367
0,word_unigram_min_df_2,word,"(1, 1)",2,10000,0.95,True,l2,987,710,3812,700770,0.005440,0.994560
1,word_unigram_bigram_min_df_2,word,"(1, 2)",2,10000,0.95,True,l2,987,1198,5269,1182426,0.004456,0.995544
3,word_unigram_min_df_5,word,"(1, 1)",5,10000,0.95,True,l2,987,213,2489,210231,0.011839,0.988161
4,word_unigram_bigram_min_df_5,word,"(1, 2)",5,10000,0.95,True,l2,987,267,2893,263529,0.010978,0.989022


Depois da comparacao, esta celula separa a configuracao baseline. A baseline atual usa `char_wb`, `ngram_range=(3, 5)`, `min_df=2` e `max_features=30000`, uma escolha adequada para descricoes fiscais com abreviacoes, marcas, erros de digitacao e variacoes curtas.

In [5]:
baseline_vectorizer = vectorizers[BASELINE_TFIDF_CONFIG_NAME]
baseline_matrix = matrices[BASELINE_TFIDF_CONFIG_NAME]

display(summary[summary["name"] == BASELINE_TFIDF_CONFIG_NAME])
display(vocabulary_samples[vocabulary_samples["config_name"] == BASELINE_TFIDF_CONFIG_NAME].head(40))

baseline_matrix.shape

,name,analyzer,ngram_range,min_df,max_features,max_df,sublinear_tf,norm,document_count,feature_count,non_zero_values,total_cells,density,sparsity
2,char_wb_3_5_min_df_2,char_wb,"(3, 5)",2,30000,0.95,True,l2,987,6727,47599,6639549,0.007169,0.992831


,config_name,sample_type,description,frequency,rank,term,value
166,char_wb_3_5_min_df_2,vocabulary_sample,NaN,NaN,1,1,NaN
167,char_wb_3_5_min_df_2,vocabulary_sample,NaN,NaN,2,10,NaN
168,char_wb_3_5_min_df_2,vocabulary_sample,NaN,NaN,3,10,NaN
169,char_wb_3_5_min_df_2,vocabulary_sample,NaN,NaN,4,100,NaN
170,char_wb_3_5_min_df_2,vocabulary_sample,NaN,NaN,5,100,NaN
171,char_wb_3_5_min_df_2,vocabulary_sample,NaN,NaN,6,100g,NaN
172,char_wb_3_5_min_df_2,vocabulary_sample,NaN,NaN,7,102,NaN
173,char_wb_3_5_min_df_2,vocabulary_sample,NaN,NaN,8,102,NaN
174,char_wb_3_5_min_df_2,vocabulary_sample,NaN,NaN,9,11,NaN
175,char_wb_3_5_min_df_2,vocabulary_sample,NaN,NaN,10,110,NaN


(987, 6727)

## Inspecionar documentos vetorizados

Esta etapa permite escolher uma configuracao de TF-IDF e verificar, documento a documento, quais termos receberam maior peso. A ideia e aproximar a matriz esparsa das descricoes legiveis para avaliar se a representacao esta capturando sinais uteis ou apenas fragmentos ruidosos.

In [6]:
BASELINE_TFIDF_CONFIG_NAME = "word_unigram_bigram_min_df_5"

TFIDF_CONFIG_TO_INSPECT = BASELINE_TFIDF_CONFIG_NAME
DOCUMENTS_TO_INSPECT = 12
TERMS_PER_DOCUMENT = 10
SAMPLE_BY = "aleatorio"  # opcoes: "frequencia", "aleatorio", "inicio"
RANDOM_STATE = 42

if TFIDF_CONFIG_TO_INSPECT not in vectorizers:
    raise ValueError(
        f"Configuracao desconhecida: {TFIDF_CONFIG_TO_INSPECT}. "
        f"Opcoes disponiveis: {sorted(vectorizers)}"
    )

inspection_vectorizer = vectorizers[TFIDF_CONFIG_TO_INSPECT]
inspection_matrix = matrices[TFIDF_CONFIG_TO_INSPECT]
inspection_feature_names = inspection_vectorizer.get_feature_names_out()

if SAMPLE_BY == "frequencia":
    inspection_rows = modeling_items.nlargest(DOCUMENTS_TO_INSPECT, "frequencia")
elif SAMPLE_BY == "aleatorio":
    inspection_rows = modeling_items.sample(
        min(DOCUMENTS_TO_INSPECT, len(modeling_items)),
        random_state=RANDOM_STATE,
    )
elif SAMPLE_BY == "inicio":
    inspection_rows = modeling_items.head(DOCUMENTS_TO_INSPECT)
else:
    raise ValueError('SAMPLE_BY deve ser "frequencia", "aleatorio" ou "inicio"')

inspection_records = []
for _, row in inspection_rows.iterrows():
    row_vector = inspection_matrix.getrow(int(row["descricao_id"]))
    top_terms = []

    if row_vector.nnz:
        local_order = row_vector.data.argsort()[::-1][:TERMS_PER_DOCUMENT]
        top_terms = [
            f"{inspection_feature_names[row_vector.indices[position]]} ({row_vector.data[position]:.3f})"
            for position in local_order
        ]

    inspection_records.append(
        {
            "config_name": TFIDF_CONFIG_TO_INSPECT,
            "descricao_id": int(row["descricao_id"]),
            "descricao_normalizada": row["descricao_normalizada"],
            "frequencia": int(row["frequencia"]),
            "termos_tfidf_mais_fortes": ", ".join(top_terms),
            "termos_com_peso": int(row_vector.nnz),
        }
    )

display(summary[summary["name"] == TFIDF_CONFIG_TO_INSPECT])
pd.DataFrame.from_records(inspection_records)


,name,analyzer,ngram_range,min_df,max_features,max_df,sublinear_tf,norm,document_count,feature_count,non_zero_values,total_cells,density,sparsity
4,word_unigram_bigram_min_df_5,word,"(1, 2)",5,10000,0.95,True,l2,987,267,2893,263529,0.010978,0.989022


,config_name,descricao_id,descricao_normalizada,frequencia,termos_tfidf_mais_fortes,termos_com_peso
0,word_unigram_bigram_min_df_5,309,coxinha carne gde kg,1,"coxinha (0.702), carne (0.622), kg (0.346)",3
1,word_unigram_bigram_min_df_5,139,beb refrig coca cola 2 5,1,"coca cola (0.528), cola (0.528), coca (0.517), beb (0.418)",4
2,word_unigram_bigram_min_df_5,499,lav louca limpol liq cristal 500ml,1,"cristal (0.545), lav (0.509), liq (0.499), 500ml (0.442)",4
3,word_unigram_bigram_min_df_5,854,saco lixo valore 15l c 20 un,2,"saco (0.868), un (0.496)",2
4,word_unigram_bigram_min_df_5,88,banana prata reser canyons organica 800g,8,,0
5,word_unigram_bigram_min_df_5,398,file coxa scx nat c pele cong iqf 1kg,2,"coxa scx (0.383), scx (0.373), coxa (0.345), nat (0.339), cong iqf (0.334), file (0.330), iqf (0.330), cong (0.276),...",9
6,word_unigram_bigram_min_df_5,905,st johnsons 80g lavanda,1,"lavanda (0.633), johnsons (0.603), 80g (0.485)",3
7,word_unigram_bigram_min_df_5,107,batata palha deutschips trad 80g,1,"deutschips (0.614), 80g (0.470), batata (0.452), trad (0.445)",4
8,word_unigram_bigram_min_df_5,59,arroz branco prato fino tp1 1kg,9,"arroz branco (0.459), tp1 1kg (0.429), tp1 (0.429), branco (0.407), arroz (0.386), 1kg (0.327)",6
9,word_unigram_bigram_min_df_5,534,ling suina aurora churrasco cong 700g,2,"churrasco (0.503), suina (0.461), ling (0.453), 700g (0.446), cong (0.362)",5


## Salvar relatorios e registro de experimentos

Os arquivos em `reports/tfidf/notas-pessoais/` registram a comparacao das representacoes, amostras do vocabulario e um registro append-only dos experimentos.

Por fim, esta celula salva a evidencia do experimento. O resumo mostra o tamanho e a esparsidade de cada matriz, as amostras ajudam a inspecionar o vocabulario gerado e o registro append-only preserva metadados para reproducibilidade no TCC.

In [7]:
registry = save_tfidf_reports(
    summary,
    vocabulary_samples,
    REPORT_DIR,
    dataset_name=DATASET_NAME,
    period_label=PERIOD_LABEL,
)

display(registry.tail())
sorted(path.relative_to(PROJECT_ROOT) for path in REPORT_DIR.glob("*.csv"))

,experiment_id,created_at,stage,dataset_name,period_label,document_count,config_name,parameters_json,metrics_json,artifacts_json,notes
1,tfidf-notas-pessoais_semstopwords-202511-202611-char_wb_3_5_min_df_5-20261102T191014Z,2026-11-02T19:10:14Z,tfidf,notas-pessoais_semstopwords,202511-202611,987,char_wb_3_5_min_df_5,"{""analyzer"": ""char_wb"", ""max_df"": 0.95, ""max_features"": 30000, ""min_df"": 5, ""ngram_range"": ""(3, 5)"", ""norm"": ""l2"", ""...","{""density"": 0.013632933568397922, ""feature_count"": 2743, ""non_zero_values"": 36909, ""sparsity"": 0.9863670664316021}","{""summary_csv"": ""C:\\Users\\samuel.simao\\Sources\\Nofis-Classifier\\reports\\tfidf\\notas-pessoais\\tfidf_experimen...",First TF-IDF comparison for filtered market dataset.
2,tfidf-notas-pessoais_semstopwords-202511-202611-word_unigram_bigram_min_df_2-20261102T191014Z,2026-11-02T19:10:14Z,tfidf,notas-pessoais_semstopwords,202511-202611,987,word_unigram_bigram_min_df_2,"{""analyzer"": ""word"", ""max_df"": 0.95, ""max_features"": 10000, ""min_df"": 2, ""ngram_range"": ""(1, 2)"", ""norm"": ""l2"", ""sub...","{""density"": 0.004456092812573472, ""feature_count"": 1198, ""non_zero_values"": 5269, ""sparsity"": 0.9955439071874266}","{""summary_csv"": ""C:\\Users\\samuel.simao\\Sources\\Nofis-Classifier\\reports\\tfidf\\notas-pessoais\\tfidf_experimen...",First TF-IDF comparison for filtered market dataset.
3,tfidf-notas-pessoais_semstopwords-202511-202611-word_unigram_bigram_min_df_5-20261102T191014Z,2026-11-02T19:10:14Z,tfidf,notas-pessoais_semstopwords,202511-202611,987,word_unigram_bigram_min_df_5,"{""analyzer"": ""word"", ""max_df"": 0.95, ""max_features"": 10000, ""min_df"": 5, ""ngram_range"": ""(1, 2)"", ""norm"": ""l2"", ""sub...","{""density"": 0.010977918938712628, ""feature_count"": 267, ""non_zero_values"": 2893, ""sparsity"": 0.9890220810612874}","{""summary_csv"": ""C:\\Users\\samuel.simao\\Sources\\Nofis-Classifier\\reports\\tfidf\\notas-pessoais\\tfidf_experimen...",First TF-IDF comparison for filtered market dataset.
4,tfidf-notas-pessoais_semstopwords-202511-202611-word_unigram_min_df_2-20261102T191014Z,2026-11-02T19:10:14Z,tfidf,notas-pessoais_semstopwords,202511-202611,987,word_unigram_min_df_2,"{""analyzer"": ""word"", ""max_df"": 0.95, ""max_features"": 10000, ""min_df"": 2, ""ngram_range"": ""(1, 1)"", ""norm"": ""l2"", ""sub...","{""density"": 0.005439730582074005, ""feature_count"": 710, ""non_zero_values"": 3812, ""sparsity"": 0.994560269417926}","{""summary_csv"": ""C:\\Users\\samuel.simao\\Sources\\Nofis-Classifier\\reports\\tfidf\\notas-pessoais\\tfidf_experimen...",First TF-IDF comparison for filtered market dataset.
5,tfidf-notas-pessoais_semstopwords-202511-202611-word_unigram_min_df_5-20261102T191014Z,2026-11-02T19:10:14Z,tfidf,notas-pessoais_semstopwords,202511-202611,987,word_unigram_min_df_5,"{""analyzer"": ""word"", ""max_df"": 0.95, ""max_features"": 10000, ""min_df"": 5, ""ngram_range"": ""(1, 1)"", ""norm"": ""l2"", ""sub...","{""density"": 0.011839357658956101, ""feature_count"": 213, ""non_zero_values"": 2489, ""sparsity"": 0.988160642341044}","{""summary_csv"": ""C:\\Users\\samuel.simao\\Sources\\Nofis-Classifier\\reports\\tfidf\\notas-pessoais\\tfidf_experimen...",First TF-IDF comparison for filtered market dataset.


[WindowsPath('reports/tfidf/notas-pessoais/experiment_registry.csv'),
 WindowsPath('reports/tfidf/notas-pessoais/tfidf_experiments_summary.csv'),
 WindowsPath('reports/tfidf/notas-pessoais/tfidf_vocabulary_samples.csv')]